In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import confusion_matrix

In [4]:
df = pd.read_csv("model_predictions.csv")

df.head()

,true,conv_pred,swin_pred,siglip_pred,ensemble_pred,conv_correct,swin_correct,siglip_correct,ensemble_correct,conv_conf,...,siglip_Recyclable,ensemble_Recyclable,conv_Electronic,swin_Electronic,siglip_Electronic,ensemble_Electronic,conv_Organic,swin_Organic,siglip_Organic,ensemble_Organic
0,0,0,0,0,0,True,True,True,True,0.904741,...,0.999995,0.936263,0.076859,0.037681,5.325263e-07,0.050789,0.018400,0.010853,4.919860e-06,0.012948
1,0,0,0,0,0,True,True,True,True,0.913278,...,0.999999,0.939828,0.071467,0.037737,4.655838e-07,0.048387,0.015255,0.011443,8.943485e-07,0.011785
2,0,0,0,0,0,True,True,True,True,0.914421,...,0.999996,0.938042,0.069362,0.035056,5.198387e-07,0.046287,0.016217,0.019471,3.058247e-06,0.015670
3,0,0,0,0,0,True,True,True,True,0.949260,...,0.993650,0.943963,0.036333,0.037032,6.271023e-03,0.033026,0.014407,0.038414,7.912327e-05,0.023011
4,0,0,0,0,0,True,True,True,True,0.949947,...,0.998107,0.955242,0.038581,0.036502,8.320664e-07,0.033057,0.011472,0.014678,1.892435e-03,0.011701


In [5]:
print(df.shape)

df.info()

(24853, 26)
<class 'pandas.DataFrame'>
RangeIndex: 24853 entries, 0 to 24852
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   true                 24853 non-null  int64  
 1   conv_pred            24853 non-null  int64  
 2   swin_pred            24853 non-null  int64  
 3   siglip_pred          24853 non-null  int64  
 4   ensemble_pred        24853 non-null  int64  
 5   conv_correct         24853 non-null  bool   
 6   swin_correct         24853 non-null  bool   
 7   siglip_correct       24853 non-null  bool   
 8   ensemble_correct     24853 non-null  bool   
 9   conv_conf            24853 non-null  float64
 10  swin_conf            24853 non-null  float64
 11  siglip_conf          24853 non-null  float64
 12  ensemble_conf        24853 non-null  float64
 13  agreement            24853 non-null  str    
 14  conv_Recyclable      24853 non-null  float64
 15  swin_Recyclable      24853 non-null

In [6]:
summary = pd.DataFrame({

    "Model":[
        "ConvNeXt",
        "Swin",
        "SigLIP",
        "Ensemble"
    ],

    "Accuracy":[

        df.conv_correct.mean(),

        df.swin_correct.mean(),

        df.siglip_correct.mean(),

        df.ensemble_correct.mean()

    ]

})

summary.sort_values(
    "Accuracy",
    ascending=False
)

,Model,Accuracy
3,Ensemble,0.981089
0,ConvNeXt,0.973645
1,Swin,0.973404
2,SigLIP,0.957993


In [7]:
agreement = (
    df
    .groupby("agreement")
    .agg(

        Samples=("true","size"),

        Accuracy=("ensemble_correct","mean")

    )

    .sort_values(
        "Samples",
        ascending=False
    )

)

agreement["Accuracy"]*=100

agreement

,Samples,Accuracy
agreement,,
3 Models Agree,23388,99.234650
2 Models Agree,1441,80.777238
All Different,24,41.666667


In [8]:
pd.crosstab(

    df.agreement,

    df.ensemble_correct,

    margins=True

)

ensemble_correct,False,True,All
agreement,,,
2 Models Agree,277,1164,1441
3 Models Agree,179,23209,23388
All Different,14,10,24
All,470,24383,24853


In [9]:
cm = confusion_matrix(

    df.true,

    df.ensemble_pred

)

pd.DataFrame(

    cm,

    index=[
        "True_Recyclable",
        "True_Electronic",
        "True_Organic"
    ],

    columns=[
        "Pred_Recyclable",
        "Pred_Electronic",
        "Pred_Organic"
    ]

)

,Pred_Recyclable,Pred_Electronic,Pred_Organic
True_Recyclable,9204,54,208
True_Electronic,24,3898,8
True_Organic,166,10,11281


In [10]:
difficulty = pd.DataFrame({

    "Total":df.true.value_counts().sort_index(),

    "Wrong":df[
        ~df.ensemble_correct
    ].true.value_counts().sort_index()

})

difficulty.index=[

    "Recyclable",

    "Electronic",

    "Organic"

]

difficulty["Error Rate"]=(
    difficulty.Wrong/
    difficulty.Total
)

In [11]:
wrong = df[
    ~df.ensemble_correct
]

wrong.shape
pd.DataFrame({

    "Rescue":[

        wrong.conv_correct.sum(),

        wrong.swin_correct.sum(),

        wrong.siglip_correct.sum()

    ]

},

index=[

    "ConvNeXt",

    "Swin",

    "SigLIP"

]).sort_values(

    "Rescue",

    ascending=False

)


,Rescue
SigLIP,129
ConvNeXt,108
Swin,99


In [12]:
confidence = pd.DataFrame({

    "Correct":[

        df.loc[df.conv_correct,"conv_conf"].mean(),

        df.loc[df.swin_correct,"swin_conf"].mean(),

        df.loc[df.siglip_correct,"siglip_conf"].mean(),

        df.loc[df.ensemble_correct,"ensemble_conf"].mean()

    ],

    "Wrong":[

        df.loc[~df.conv_correct,"conv_conf"].mean(),

        df.loc[~df.swin_correct,"swin_conf"].mean(),

        df.loc[~df.siglip_correct,"siglip_conf"].mean(),

        df.loc[~df.ensemble_correct,"ensemble_conf"].mean()

    ]

},

index=[

    "ConvNeXt",

    "Swin",

    "SigLIP",

    "Ensemble"

])

confidence

,Correct,Wrong
ConvNeXt,0.932987,0.775732
Swin,0.937317,0.771106
SigLIP,0.982620,0.822002
Ensemble,0.931763,0.691674


In [13]:
df[

    [

        "conv_conf",

        "swin_conf",

        "siglip_conf",

        "ensemble_conf"

    ]

].describe()

,conv_conf,swin_conf,siglip_conf,ensemble_conf
count,24853.000000,24853.000000,24853.000000,24853.000000
mean,0.928843,0.932896,0.975873,0.927222
std,0.063775,0.064916,0.076042,0.078982
min,0.389185,0.388512,0.380890,0.397019
25%,0.928759,0.931178,0.995159,0.936114
50%,0.936789,0.941963,0.999849,0.943693
75%,0.948575,0.952934,0.999999,0.954289
max,0.998352,0.999357,1.000000,0.995229


In [14]:
(df.conv_pred==df.swin_pred).mean()

np.float64(0.9678912002575142)

In [15]:
(df.conv_pred==df.siglip_pred).mean()

np.float64(0.9547740715406591)

In [16]:
(df.swin_pred==df.siglip_pred).mean()

np.float64(0.9584758379270109)

In [17]:
hard = df[

    (~df.conv_correct)&
    (~df.swin_correct)&
    (~df.siglip_correct)

]

hard.shape

(189, 26)

In [18]:
easy = df[

    (df.conv_correct)&
    (df.swin_correct)&
    (df.siglip_correct)

]

easy.shape

(23209, 26)

In [19]:
wrong.groupby(

    [

        "true",

        "ensemble_pred"

    ]

).size().sort_values(

    ascending=False

)

true  ensemble_pred
0     2                208
2     0                166
0     1                 54
1     0                 24
2     1                 10
1     2                  8
dtype: int64

In [20]:
wrong.groupby(

    [

        "conv_correct",

        "swin_correct",

        "siglip_correct"

    ]

).size().sort_values(

    ascending=False

)

conv_correct  swin_correct  siglip_correct
False         False         False             189
True          False         False              83
False         False         True               75
              True          False              68
                            True               30
True          False         True               24
              True          False               1
dtype: int64